### RaschPy PCM worked example

This notebook works through a sample Rasch analysis of a simulated data set (1,000 persons, 10 items - 4 with a maximum score of 5 and 6 with a maximum score of 3, and no missing data), taking you through the relevant commands step by step, with notes before each cell. Relevant outputs will appear below each cell.

Import the modules and set the working directory (here called `my_working_directory`) to where you want to save your output files.

In [ ]:
import raschpy as rp
import os

os.chdir('my_working_directory')

**A note on data validation**

Every model constructor validates the item-response network automatically at instantiation (`validate=True` by default) and warns if it's disconnected — if there's no chain of persons and items linking every item to every other, the resulting item locations aren't on a common scale, even though `calibrate()` will still run without error. This is worth seeing happen once. Here we build a small, deliberately disconnected data set: two groups of persons who each only answer a disjoint set of items, with no item shared between the groups to link them:

In [ ]:
sim_disconnected = rp.PCM_Sim(no_of_items=10, no_of_persons=80, max_score_vector=[4] * 10, seed=108)
responses_disconnected = sim_disconnected.responses.copy()
responses_disconnected.iloc[:40, 5:] = float('nan')   # first 40 persons: only answer items 1-5
responses_disconnected.iloc[40:, :5] = float('nan')    # last 40 persons: only answer items 6-10

broken_pcm = rp.PCM(responses_disconnected, [4] * 10)   # raises a UserWarning

`connectivity_status` records the diagnosis, including which items ended up in which isolated sub-group:

In [ ]:
broken_pcm.connectivity_status

The rest of this notebook uses a single, fully-connected simulated data set, so this warning won't come up again.

Simulate a data set. Passing `seed=42` makes the simulation fully reproducible — rerunning this notebook will always generate the same data set. 10 items, 1,000 persons, 20% missing data (so each person responds to ~8 items and each item has ~800 responses).

In [ ]:
max_score_vector = [5, 5, 5, 5, 3, 3, 3, 3, 3, 3]
sim = rp.PCM_Sim(no_of_items=10, no_of_persons=1000, max_score_vector=max_score_vector, missing=0.2, seed=42)
sim.responses.to_csv('pcm_scores.csv')

If you have your own response data saved to a CSV file instead of simulating it, use `loadup_pcm()` to load and validate it. Demonstrated here by reloading the file we just saved:

In [ ]:
data, invalid_responses = rp.loadup_pcm('pcm_scores.csv', max_score_vector=max_score_vector)

Check the data - view the first two lines

In [ ]:
data.head(2)

Check for any invalid responses (not usable for estimation purposes and excluded)

In [ ]:
invalid_responses

Create a PCM object. Passing the simulation object `sim` directly (rather than the reloaded `data`) attaches the generating parameters under `pcm.generating`, which lets us check parameter recovery further down, and also means `max_score_vector` doesn't need to be supplied again. If you're analysing your own data, pass a DataFrame and `max_score_vector` instead (e.g. `rp.PCM(data, max_score_vector)`).

In [ ]:
pcm = rp.PCM(sim)

Generate item estimates. The `%%time` "magic function" returns the time taken to run the cell contents (algorithm run time in this case).

In [ ]:
%%time
pcm.calibrate()

Check the central item location estimates - view the first two items

In [ ]:
pcm.items.head(2)

Since this is simulated data, we know the true generating item locations (`slm.generating.items`) and can check how closely the calibration recovered them. The helper below plots generating vs. estimated values, with an identity line (dashed dark red) and a fitted regression line (dashed red) — the closer the points hug the identity line, the better the recovery. Also displays the Pearson correlation, SD ratio, regression coefficient and RMSE:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

def recovery_plot(generating, estimated, label, filename):
    x, y = np.asarray(generating), np.asarray(estimated)
    fig, ax = plt.subplots()
    ax.scatter(x, y, alpha=0.6)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], color='DarkRed', label='Identity')
    m, b = np.polyfit(x, y, 1)
    ax.plot([lo, hi], [m * lo + b, m * hi + b], color='Red', linestyle='--', label='Regression')
    ax.set_xlabel(f'Generating {label}')
    ax.set_ylabel(f'Estimated {label}')
    ax.set_aspect('equal')
    ax.legend()
    plt.savefig(filename)
    plt.show()
    print(f'{label} Pearson correlation:              {round(np.corrcoef(x, y)[0, 1], 3)}')
    print(f'{label} SD ratio (estimated / generating): {round(y.std() / x.std(), 3)}')
    print(f'{label} regression coefficient:            {round(m, 3)}')
    print(f'{label} RMSE:                              {round(np.sqrt(((x - y) ** 2).mean()), 3)}')

recovery_plot(sim.items, pcm.items, 'item location', 'my_pcm_item_recovery.png')

Generate a table of item statistics (and check run time), and save to file

In [ ]:
%%time
pcm.item_stats_df(full=True)
pcm.item_stats.to_csv('pcm_item_stats.csv')

Check the item statistics table

In [ ]:
pcm.item_stats

Generate a table of threshold statistics (and check run time), and save to file

In [ ]:
%%time
pcm.threshold_stats_df(full=True)
pcm.threshold_stats.to_csv('pcm_threshold_stats_centred.csv')
pcm.threshold_stats_uncentred.to_csv('pcm_threshold_stats_uncentred.csv')

Check the centred threshold statistics table - view the first two thresholds

In [ ]:
pcm.threshold_stats.head(2)

Check the uncentred threshold statistics table - view the first two items

In [ ]:
pcm.threshold_stats_uncentred.head(2)

And the same recovery check for the uncentred threshold structure, against `sim.thresholds_uncentred`. Since thresholds are per-item, we flatten (item, threshold) pairs into a single series with `.unstack()` first:

In [ ]:
orig_thresholds = sim.thresholds_uncentred.unstack().dropna()
est_thresholds = pcm.thresholds_uncentred.unstack().dropna()
recovery_plot(orig_thresholds, est_thresholds, 'uncentred threshold', 'my_pcm_threshold_recovery.png')

Generate a table of person statistics (and check run time), and save to file

In [ ]:
%%time
pcm.person_stats_df(full=True)
pcm.person_stats.to_csv('pcm_person_stats.csv')

Check the person statistics table - view the first ten persons with `.head(10)`

In [ ]:
pcm.person_stats.head(10)

And the same recovery check for person locations, against `sim.persons`:

In [ ]:
recovery_plot(sim.persons, pcm.persons, 'person location', 'my_pcm_person_recovery.png')

Generate a table of test-level statistics (and check run time), and save to file

In [ ]:
%%time
pcm.test_stats_df()
pcm.test_stats.to_csv('pcm_test_stats.csv')

Check the test statistics table

In [ ]:
pcm.test_stats

Run a residual correlation analysis (and check run time), and save relevant output to file

In [ ]:
%%time
pcm.res_corr_analysis()
pcm.residual_correlations.to_csv('pcm_residual_correlations.csv')
pcm.loadings.to_csv('pcm_loadings.csv')

View the table of pairwise standard residual correlations

In [ ]:
round(pcm.residual_correlations, 3)

View the item loadings on the first principal component of the pairwise standard residual correlations (dimensionality test)

In [ ]:
round(pcm.loadings['PC 1'], 3)

Produce an item characteristic curve (item response function) curve for Item 2, with observed category means plotted and the thresholds marked

In [ ]:
pcm.icc('Item_2', title='ICC for Item 2', obs=True, thresh_lines=True, cat_highlight=2, xmin=-5, xmax=5, filename='my_pcm_icc')

Produce category response curves for Item 2, with the central item location marked

In [ ]:
pcm.crcs('Item_2', central_location=True, obs=[2], xmin=-5, xmax=5, filename='my_pcm_crcs')

Produce threshold characteristic curves for Item 2, with observed category means plotted for threshold 2 and the central item location marked

In [ ]:
pcm.threshold_ccs('Item_2', central_location=True, obs=[2], cat_highlight=4, xmin=-5, xmax=5, filename='my_pcm_threshold_ccs')

Produce an item information function curve for Item 2

In [ ]:
pcm.iic('Item_2', point_info_lines=[0], point_info_labels=True, title='Information for Item 2', xmin=-5, xmax=5, filename='my_pcm_iic')

Produce a test characteristic curve (test response function), with person locations corresponding to scores of 15 and 25 plotted.

In [ ]:
pcm.tcc(score_lines=[15, 25], score_labels=True, filename='my_pcm_tcc')

Produce a test information curve

In [ ]:
pcm.test_info(point_info_lines=[0], point_info_labels=True, filename='my_pcm_test_info_curve')

Produce a test CSEM (conditional standard error of measurement) curve, with the CSEM corresponding to a person location of -3 plotted

In [ ]:
pcm.test_csem(point_csem_lines=[-3], point_csem_labels=True, ymax=1, filename='my_pcm_csem_curve')

Produce a histogram of standardised residuals, with a normal distribution curve overlaid

In [ ]:
pcm.std_residuals_plot(bin_width=0.6, normal=True, filename='my_pcm_std_residuals_plot')